In [ ]:
import re
import csv
import json
import time
import requests
from bs4 import BeautifulSoup

# --- Config: what we're scraping -------------------------------------------
BASE = "https://www.dbl.id/match/schedule"
PARAMS_BASE = {"season_year": 2026, "series_id": 2, "region_id": 2}  # East Java Series, South Region
HEADERS = {"User-Agent": "Mozilla/5.0 (research script; contact: you@example.com)"}

MUST_DEBUG_DATES = {"2022-09-06", "2026-08-14"}

FALLBACK_DATES = {
    2019: ["2019-08-31","2019-09-01","2019-09-02","2019-09-03","2019-09-04","2019-09-05",
           "2019-09-06","2019-09-07","2019-09-08","2019-09-09","2019-09-10","2019-09-11",
           "2019-09-14","2019-09-15","2019-09-17","2019-09-18","2019-09-20"],
    2022: ["2022-08-30","2022-08-31","2022-09-01","2022-09-02","2022-09-03","2022-09-04",
           "2022-09-05","2022-09-06","2022-09-07","2022-09-08","2022-09-09","2022-09-10",
           "2022-09-13","2022-09-14","2022-09-15","2022-09-16","2022-09-17","2022-09-19"],
    2023: ["2023-08-03","2023-08-04","2023-08-05","2023-08-06","2023-08-07","2023-08-08",
           "2023-08-09","2023-08-10","2023-08-11","2023-08-12","2023-08-13","2023-08-14",
           "2023-08-15","2023-08-17","2023-08-18","2023-08-19","2023-08-20","2023-08-21","2023-08-23"],
    2024: ["2024-09-04","2024-09-05","2024-09-06","2024-09-07","2024-09-08","2024-09-09",
           "2024-09-10","2024-09-11","2024-09-12","2024-09-13","2024-09-14","2024-09-15",
           "2024-09-16","2024-09-17","2024-09-18","2024-09-20","2024-09-21","2024-09-22",
           "2024-09-23","2024-09-24","2024-09-26"],
    2025: ["2025-08-12","2025-08-13","2025-08-14","2025-08-15","2025-08-16","2025-08-17",
           "2025-08-18","2025-08-19","2025-08-20","2025-08-21","2025-08-22","2025-08-23",
           "2025-08-24","2025-08-25","2025-08-26","2025-08-28","2025-08-29","2025-08-30",
           "2025-08-31","2025-09-06","2025-09-07"],
    2026: ["2026-08-13","2026-08-14","2026-08-15","2026-08-16","2026-08-17","2026-08-18",
           "2026-08-19","2026-08-20","2026-08-21","2026-08-22","2026-08-23","2026-08-24",
           "2026-08-25","2026-08-26","2026-08-27","2026-08-29","2026-08-30","2026-08-31",
           "2026-09-01","2026-09-02","2026-09-04"],
}


def discover_all_dates(session):
    known = set()
    for yr_dates in FALLBACK_DATES.values():
        known.update(yr_dates)

    try:
        r = session.get(BASE, params=PARAMS_BASE, headers=HEADERS, timeout=20)
        r.raise_for_status()
        discovered = set(re.findall(r"date=(\d{4}-\d{2}-\d{2})", r.text))
        new_ones = discovered - known
        if new_ones:
            print(f"Found {len(new_ones)} additional date(s) not in the "
                  f"hardcoded list: {sorted(new_ones)}")
        else:
            print("Live discovery found nothing new (expected — dbl.id's "
                  "date tabs are JS-rendered and won't show up here). "
                  f"Using the {len(known)} known verified dates.")
        known.update(new_ones)
    except Exception as e:
        print(f"Live discovery failed ({e}); using the {len(known)} known "
              "verified dates only.")

    return sorted(known)


def _get_card(link, max_depth):
    """Walk up from a match-detail link until the container text looks like
    a whole card. Identical to the original script's logic."""
    card = link
    for _ in range(max_depth):
        if card.parent is None:
            break
        card = card.parent
        text = card.get_text(" | ", strip=True)
        if text.count("|") >= 6 and re.search(r"\d{1,2}:\d{2}", text):
            break
    return card


def _try_strict(lines):
    """The ORIGINAL script's exact extraction rules, unchanged. Returns
    (nums, names, meta_line) or None if it doesn't match this shape."""
    nums, meta_line, potential_names = [], "", []
    for l in lines:
        if re.fullmatch(r"\d{1,3}", l):
            nums.append(l)
        elif "•" in l:
            meta_line = l
        else:
            potential_names.append(l)
    names = [l for l in potential_names if len(l) > 2]

    if len(nums) < 2 or len(names) < 2 or not meta_line:
        return None
    return nums, names, meta_line


def _try_loose(lines):
    """Fallback ONLY used when _try_strict fails on a real match-detail
    link. Widens the same three things one at a time: separator character,
    combined 'NN-NN' score format, and the name-length filter."""
    nums, meta_line, potential_names = [], "", []
    bullets = "•·‣∙◦"
    for l in lines:
        if re.fullmatch(r"\d{1,3}", l):
            nums.append(l)
        elif any(ch in l for ch in bullets):
            if not meta_line:
                meta_line = l
        else:
            potential_names.append(l)

    if len(nums) < 2:
        for l in lines:
            combo = re.search(r"\b(\d{1,3})\s*[-–:]\s*(\d{1,3})\b", l)
            if combo:
                nums = [combo.group(1), combo.group(2)]
                break

    names = [l for l in potential_names if l and not re.fullmatch(r"[\W_]+", l)]

    if len(nums) < 2 or len(names) < 2 or not meta_line:
        return None
    return nums, names, meta_line


def _split_meta(meta_line):
    parts = [p.strip() for p in meta_line.split("•")] if "•" in meta_line else \
            [p.strip() for p in re.split(r"[·‣∙◦]", meta_line)]
    parts = [p for p in parts if p]
    category = parts[0] if parts else ""
    time_wib = next((p for p in parts if re.search(r"\d{1,2}:\d{2}", p)), "")
    venue = parts[-2] if len(parts) >= 2 else ""
    round_name = parts[-1] if len(parts) >= 1 else ""
    return category, time_wib, venue, round_name


# --- NEW: bug fix for days that silently reported 0 games ------------------
#
# Root cause (confirmed from the notebook's own run log): for every single
# day that came back with 0 games AND 0 review rows, the diagnostic check
# `"/match/detail/" not in html` never fired — meaning the substring WAS
# present in the raw response. But `soup.select('a[href*="/match/detail/"]')`
# found zero real <a> tags. The only way both of those are true is that the
# match links exist in the page as embedded JSON (the data Next.js/React
# uses to hydrate the page client-side) rather than as server-rendered
# anchor tags that BeautifulSoup can select. The old code only ever looked
# for <a> tags, so those games vanished with no trace and no review entry.
#
# Fix: after the normal anchor-based pass, find every game id that appears
# ANYWHERE in the page via regex, diff it against the ids we already
# captured from anchors, and for anything missing, try to pull the matching
# object out of a `__NEXT_DATA__` (or any) JSON <script> blob. We don't
# guess field names — we don't know dbl.id's exact JSON schema and can't
# fetch the live site from this sandbox to check — so instead we dump the
# raw matched JSON object (or a raw HTML snippet if no JSON blob is found)
# into the review CSV. That turns a silent "0 games" into a visible,
# debuggable review row you can inspect, instead of losing the game
# entirely.

GAME_ID_RE = re.compile(r"/match/detail/(\d+)")


def _find_regex_game_ids(html):
    """Every game id that appears anywhere in the raw HTML/JS, not just
    inside <a> tags."""
    return set(GAME_ID_RE.findall(html))


def _find_json_blobs(soup):
    """Return parsed JSON payloads from every <script type="application/json">
    tag (this covers Next.js's __NEXT_DATA__ and similar hydration blobs).
    Skips anything that isn't valid JSON rather than raising."""
    blobs = []
    for tag in soup.find_all("script"):
        if not tag.string:
            continue
        stype = (tag.get("type") or "").lower()
        tid = (tag.get("id") or "")
        if "json" not in stype and tid != "__NEXT_DATA__":
            continue
        try:
            blobs.append(json.loads(tag.string))
        except Exception:
            continue
    return blobs


def _walk_json_for_ids(node, wanted_ids, found):
    """Recursively walk a JSON-like structure looking for any dict that
    references one of `wanted_ids`, either via an 'id' field or via a
    string value containing '/match/detail/<id>'. Populates `found`
    in place as {id: matching_dict}."""
    if isinstance(node, dict):
        node_id = str(node.get("id")) if node.get("id") is not None else None
        if node_id in wanted_ids and node_id not in found:
            found[node_id] = node
        else:
            for v in node.values():
                if isinstance(v, str):
                    m = GAME_ID_RE.search(v)
                    if m and m.group(1) in wanted_ids and m.group(1) not in found:
                        found[m.group(1)] = node
        for v in node.values():
            _walk_json_for_ids(v, wanted_ids, found)
    elif isinstance(node, list):
        for item in node:
            _walk_json_for_ids(item, wanted_ids, found)


def _raw_html_snippet_for_id(html, game_id, radius=600):
    """Grab raw text around a game id's first occurrence in the HTML, as a
    last-resort debug aid when no JSON blob explains it either."""
    m = re.search(re.escape(f"/match/detail/{game_id}"), html)
    if not m:
        return ""
    start = max(0, m.start() - radius)
    end = min(len(html), m.end() + radius)
    return html[start:end]


def parse_day(html, date_str):
    """Extract every game on one schedule/results day-page.

    For each match-detail link: try the ORIGINAL strict rules first (this
    is what already worked for ~95 of your 117 days — untouched). Only if
    that fails do we try a looser fallback. Only if BOTH fail does the card
    go to the review list, with its raw text, instead of vanishing.

    NEW: after the anchor pass, also recover any game ids that exist in the
    page's embedded JSON/script data but never appeared as an <a> tag (see
    the big comment above `_find_regex_game_ids`). These are added to the
    review list too, so a day with hydration-only games shows up as
    "0 games (+N need review)" instead of a silent, indistinguishable "0
    games".
    """
    soup = BeautifulSoup(html, "html.parser")

    # dbl.id renders each game's card TWICE (once for a grid view, once for
    # a list view) — both link to the same /match/detail/<id>. We must try
    # parsing EVERY occurrence and keep whichever one succeeds; deduping by
    # game_id before parsing (like an earlier version of this script did)
    # means that if the first occurrence happens to be the "thin" one, the
    # second, parseable occurrence never even gets tried. So: collect every
    # attempt per game_id first, decide success/failure after the fact.
    attempts_by_id = {}  # game_id -> list of (record_or_None, raw_text)

    for link in soup.select('a[href*="/match/detail/"]'):
        href = link.get("href", "")
        m = re.search(r"/match/detail/(\d+)", href)
        if not m:
            continue
        game_id = m.group(1)

        card = _get_card(link, max_depth=6)
        text = card.get_text("\n", strip=True)
        lines = [l for l in text.split("\n") if l.strip()]

        result = _try_strict(lines)
        used_fallback = False
        if result is None:
            result = _try_loose(lines)
            used_fallback = True

        record = None
        if result is not None:
            nums, names, meta_line = result
            category, time_wib, venue, round_name = _split_meta(meta_line)
            record = {
                "game_id": game_id,
                "date": date_str,
                "season_year": date_str[:4],
                "series": "East Java Series",
                "region": "South Region",
                "category": category,
                "round": round_name,
                "venue": venue,
                "time_wib": time_wib,
                "home_team": names[0],
                "away_team": names[1],
                "home_score": int(nums[0]),
                "away_score": int(nums[1]),
                "source_url": f"https://www.dbl.id/match/detail/{game_id}",
                "_is_boys": category.strip().lower() == "boys",
                "_used_fallback": used_fallback,
            }

        attempts_by_id.setdefault(game_id, []).append((record, text))

    games, review = [], []
    for game_id, attempts in attempts_by_id.items():
        success = next((rec for rec, _ in attempts if rec is not None), None)

        if success is None:
            # Every occurrence of this link failed to parse at all.
            review.append({
                "game_id": game_id,
                "date": date_str,
                "source_url": f"https://www.dbl.id/match/detail/{game_id}",
                "parse_status": "unparsed",
                "raw_card_text": attempts[-1][1][:400],
            })
            continue

        is_boys = success.pop("_is_boys")
        used_fallback = success.pop("_used_fallback")

        if is_boys:
            games.append(success)
        elif used_fallback:
            # Parsed fine, but category wasn't exactly "boys" via the loose
            # path (different label?) — flag rather than silently assume.
            success["parse_status"] = "non_standard_category"
            success["raw_card_text"] = ""
            review.append(success)
        # else: cleanly parsed via the strict path and clearly Girls/3x3 —
        # correctly dropped, same as the original script.

    # --- NEW: recover games that never showed up as <a> tags at all -------
    regex_ids = _find_regex_game_ids(html)
    missing_ids = regex_ids - set(attempts_by_id.keys())

    if missing_ids:
        json_blobs = _find_json_blobs(soup)
        found_in_json = {}
        for blob in json_blobs:
            _walk_json_for_ids(blob, missing_ids, found_in_json)

        for game_id in sorted(missing_ids):
            if game_id in found_in_json:
                review.append({
                    "game_id": game_id,
                    "date": date_str,
                    "source_url": f"https://www.dbl.id/match/detail/{game_id}",
                    "parse_status": "found_in_json_not_anchor",
                    "raw_card_text": json.dumps(found_in_json[game_id])[:800],
                })
            else:
                review.append({
                    "game_id": game_id,
                    "date": date_str,
                    "source_url": f"https://www.dbl.id/match/detail/{game_id}",
                    "parse_status": "found_by_regex_only",
                    "raw_card_text": _raw_html_snippet_for_id(html, game_id)[:800],
                })

    return games, review


def main():
    print("=== scrape_dbl_south_region.py — v8 (recovers games hidden in "
          "JS/JSON hydration data that v7 silently dropped) ===\n")

    session = requests.Session()
    all_dates = discover_all_dates(session)
    print(f"Found {len(all_dates)} match days to scrape.")

    all_games, all_review = [], []
    zero_game_days = []
    seen_response_hashes = {}
    snippets_saved_by_year = {}

    for i, date_str in enumerate(all_dates, 1):
        year = int(date_str[:4])
        params = dict(PARAMS_BASE, date=date_str, season_year=year)

        day_games, day_review, html = [], [], ""
        for attempt in range(4):
            try:
                r = session.get(BASE, params=params, headers=HEADERS, timeout=20)
                r.raise_for_status()
                html = r.text
            except Exception as e:
                print(f"[{i}/{len(all_dates)}] {date_str}: request FAILED ({e})")
                break

            content_hash = hash(html)
            first_seen_date = seen_response_hashes.setdefault(content_hash, date_str)
            is_suspicious_repeat = (first_seen_date != date_str)

            if is_suspicious_repeat:
                wait = 5 * (attempt + 1)
                print(f"[{i}/{len(all_dates)}] {date_str}: got the same page "
                      f"already seen for {first_seen_date} — looks rate-limited. "
                      f"Backing off {wait}s and retrying (attempt {attempt+1}/4)...")
                time.sleep(wait)
                continue

            day_games, day_review = parse_day(html, date_str)
            break

        all_games.extend(day_games)
        all_review.extend(day_review)
        note = f" (+{len(day_review)} need review)" if day_review else ""
        print(f"[{i}/{len(all_dates)}] {date_str}: {len(day_games)} games{note}")

        # NEW: the old check `"/match/detail/" not in html` used a substring
        # search over the WHOLE raw page (including embedded JS/JSON), so it
        # never fired on the very days it needed to — those pages had the
        # substring present in hydration data even though 0 anchors and 0
        # review rows came out. Check real anchors instead, so this verdict
        # actually reflects what the parser could see.
        anchors_found = len(BeautifulSoup(html, "html.parser").select(
            'a[href*="/match/detail/"]')) if html else 0

        if not day_games and not day_review:
            zero_game_days.append(date_str)
            if anchors_found == 0 and not _find_regex_game_ids(html):
                fname = f"debug_page_{date_str}.html"
                with open(fname, "w", encoding="utf-8") as dbg:
                    dbg.write(html)

                # Diagnose WHY there are no links: genuinely no games that
                # day, vs. blocked/rate-limited, vs. a broken request.
                status = r.status_code if 'r' in dir() else "?"
                size = len(html)
                lowered = html.lower()
                block_signals = [kw for kw in (
                    "captcha", "access denied", "too many requests",
                    "rate limit", "cloudflare", "just a moment",
                    "attention required", "blocked",
                ) if kw in lowered]
                has_no_games_text = any(kw in lowered for kw in (
                    "tidak ada pertandingan", "no matches", "belum ada jadwal",
                ))

                if block_signals:
                    verdict = f"LOOKS BLOCKED (found: {', '.join(block_signals)})"
                elif has_no_games_text:
                    verdict = "page explicitly says no games scheduled"
                elif size < 2000:
                    verdict = "suspiciously small response — possibly an error page"
                else:
                    verdict = "page loaded normally but has no match links or ids anywhere — likely a genuine off-day for this region"

                print(f"    -> No 'match/detail' links or ids found (HTTP {status}, {size} bytes). "
                      f"Verdict: {verdict}. Saved to {fname}.")

        if day_review:
            year_str = date_str[:4]
            already_for_year = snippets_saved_by_year.get(year_str, 0)
            should_save = (date_str in MUST_DEBUG_DATES) or (already_for_year < 2)
            if should_save:
                soup = BeautifulSoup(html, "html.parser")
                first_link = soup.select_one('a[href*="/match/detail/"]')
                if first_link is not None:
                    fname = f"debug_snippet_{date_str}.html"
                    node = first_link
                    for _ in range(8):
                        if node.parent is None:
                            break
                        node = node.parent
                    with open(fname, "w", encoding="utf-8") as dbg:
                        dbg.write(str(node.prettify()))
                    print(f"    -> Saved surrounding HTML for the unparsed card(s) to {fname}.")
                    snippets_saved_by_year[year_str] = already_for_year + 1
                else:
                    # NEW: this day's review rows came entirely from the
                    # JSON/regex recovery path (no anchors at all), so save
                    # the whole page instead of an anchor-rooted snippet.
                    fname = f"debug_page_{date_str}.html"
                    with open(fname, "w", encoding="utf-8") as dbg:
                        dbg.write(html)
                    print(f"    -> No anchors on this page; saved full page to {fname} "
                          f"(review rows for this day came from JSON/regex recovery — "
                          f"check their raw_card_text in the review CSV to see the real field names).")
                    snippets_saved_by_year[year_str] = already_for_year + 1

        time.sleep(1.5 + 0.5 * (i % 3))

    if zero_game_days:
        print(f"\n{len(zero_game_days)} of {len(all_dates)} days returned nothing at all: "
              f"{zero_game_days[:10]}{'...' if len(zero_game_days) > 10 else ''}")
        print("These had no /match/detail/ links OR ids anywhere on the page — likely genuine "
              "off-days, or still rate-limiting if debug_page_<date>.html files look "
              "identical to each other.")

    if not all_games and not all_review:
        print("No games parsed — dbl.id's markup may have changed; "
              "inspect one page's HTML and adjust parse_day().")
        return

    for g in all_games:
        g["winner"] = g["home_team"] if g["home_score"] > g["away_score"] else g["away_team"]
        g["margin"] = abs(g["home_score"] - g["away_score"])

    fieldnames = ["game_id", "date", "season_year", "series", "region", "category",
                  "round", "venue", "time_wib", "home_team", "away_team",
                  "home_score", "away_score", "winner", "margin", "source_url"]

    with open("dbl_south_region_boys_games.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(all_games)
    print(f"\nWrote {len(all_games)} Boys games to dbl_south_region_boys_games.csv")

    if all_review:
        review_fields = ["game_id", "date", "season_year", "category", "round", "venue",
                          "time_wib", "home_team", "away_team", "home_score", "away_score",
                          "source_url", "parse_status", "raw_card_text"]
        with open("dbl_south_region_needs_review.csv", "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=review_fields, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(all_review)
        print(f"Wrote {len(all_review)} games needing review to "
              f"dbl_south_region_needs_review.csv (real match links/ids the strict, "
              f"loose, AND json-recovery paths still couldn't fully parse, or "
              f"non-'boys' categories caught by the loose path). Check parse_status "
              f"'found_in_json_not_anchor' rows first — their raw_card_text shows the "
              f"real JSON field names dbl.id uses for these hydration-only games, "
              f"which is what we need to extend the parser properly.")


if __name__ == "__main__":
    main()
